# Full-match TPU v5e-1 test
Test input: JaHbat vs FC-1, September 13, 2026 (Drive file supplied by the user, 28.3 GB).
Select a **single-chip TPU v5e-1** Colab kernel in VS Code and run the first code cell. If it installs PyTorch/XLA, use VS Code's **Restart Kernel** command (or Colab's **Runtime > Restart session**) before running the cells from the top. Rerunning a cell does not restart the kernel or unload old PyTorch libraries. The notebook verifies that PyTorch/XLA sees one TPU device, installs the pinned model, downloads the exact input, and processes the entire video without cached scores. The kernel selection determines the TPU generation; the device check alone cannot identify it when Colab omits accelerator metadata. Accuracy requires ground-truth annotations. Outputs remain in the Colab runtime under `outputs/full_match_tpu_v5e_1`; download them before disconnecting.


In [ ]:
import importlib.metadata, os, subprocess, sys, json, time
from pathlib import Path
# Colab may preload torch_tpu, whose auto-loader conflicts with this XLA stack.
os.environ["TORCH_DEVICE_BACKEND_AUTOLOAD"] = "0"
# XLA 2.9's [tpu] extra pins libtpu 0.0.21, which has no Python 3.13 wheel.
# Use its Python 3.13-compatible patch release; newer libtpu breaks PJRT execution.
required = {"torch": "2.9.0", "torch-xla": "2.9.0",
            "torchvision": "0.24.0", "libtpu": "0.0.21.1"}
def installed_version(package):
    try:
        return importlib.metadata.version(package).split("+")[0]
    except importlib.metadata.PackageNotFoundError:
        return None
if any(installed_version(package) != version for package, version in required.items()):
    subprocess.run([sys.executable, "-m", "pip", "install",
                    f"torch=={required['torch']}", f"torchvision=={required['torchvision']}",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install",
                    f"torch_xla=={required['torch-xla']}",
                    f"libtpu=={required['libtpu']}"], check=True)
    raise RuntimeError("Packages installed. Use Restart Kernel in VS Code (or Runtime > Restart session in Colab), then rerun all cells.")
import torch
if torch.__version__.split("+")[0] != required["torch"]:
    raise RuntimeError(f"This kernel still has PyTorch {torch.__version__} loaded. Restart the kernel, then rerun all cells.")
try:
    import torch_xla
except ImportError as exc:
    if "undefined symbol" in str(exc):
        probe = subprocess.run([sys.executable, "-c", "import torch, torch_xla"],
                               capture_output=True, text=True)
        if probe.returncode == 0:
            raise RuntimeError("PyTorch/XLA imports in a fresh process, but this kernel has old binaries loaded. Use Restart Kernel in VS Code (or Runtime > Restart session in Colab), then rerun all cells.") from exc
    raise
import torch_xla.core.xla_model as xm
real_devices = torch_xla.real_devices()
assert len(real_devices) == 1 and str(real_devices[0]).startswith("TPU:"), (
    f"Connect a single-chip TPU v5e-1 kernel; found {real_devices}")
DEVICE = xm.xla_device()
assert DEVICE.type == "xla", f"Expected XLA device, got {DEVICE}"
assert (torch.ones(1, device=DEVICE) + 1).cpu().item() == 2, "TPU execution check failed"
accelerator = os.environ.get("TPU_ACCELERATOR_TYPE", "")
if accelerator:
    assert "v5e" in accelerator.lower() or "v5litepod-1" in accelerator.lower(), (
        f"Expected TPU v5e-1, got {accelerator}")
TPU = accelerator or str(real_devices[0])
print("Verified single TPU device:", real_devices[0])
print("Accelerator type:", accelerator or "not exposed by runtime; confirm v5e-1 in kernel settings")
print("PyTorch:", torch.__version__, "PyTorch/XLA:", torch_xla.__version__)


In [2]:
REPO = Path("/content/TEED-match-tagger")
REVISION = "ed3b0943e43646591b32407c0ad23a1951921834"
def run(*args):
    subprocess.run(list(args), check=True)
if not REPO.exists():
    run("git", "clone", "https://github.com/jmoussa99/TEED-match-tagger.git", str(REPO))
assert subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip() == "", "Remote checkout has changes; inspect before setup"
run("git", "-C", str(REPO), "checkout", REVISION)
MODEL = REPO / "private/T-DEED"
if not MODEL.exists():
    run("git", "clone", "https://github.com/arturxe2/T-DEED.git", str(MODEL))
run("git", "-C", str(MODEL), "checkout", "d953b4de8a2ad941c565e1683c5b07255cbe644e")
patch = str(REPO / "app/patches/tdeed_mps.patch")
if subprocess.run(["git", "-C", str(MODEL), "apply", "--check", patch], capture_output=True).returncode == 0:
    run("git", "-C", str(MODEL), "apply", patch)
else:
    run("git", "-C", str(MODEL), "apply", "--reverse", "--check", patch)
run(sys.executable, "-m", "pip", "install", "timm", "gdown", "pandas", "opencv-python-headless")
CKPT = MODEL / "checkpoints/SoccerNetBall/SoccerNetBall_challenge2/checkpoint_best.pt"
CKPT.parent.mkdir(parents=True, exist_ok=True)
if not CKPT.is_file():
    run(sys.executable, "-m", "gdown", "1uCWam8EhAXt3z6QQAfhRfzHITUcml438", "-O", str(CKPT))
print("Setup complete:", CKPT)


Setup complete: /content/TEED-match-tagger/private/T-DEED/checkpoints/SoccerNetBall/SoccerNetBall_challenge2/checkpoint_best.pt


In [3]:
# Exact match supplied by the user: JaHbat vs FC-1, September 13, 2026.
# Downloads directly into Colab; no 28 GB transfer through the local computer.
import shutil
import gdown
FILE_ID = "1lkoCqWRF9CEvbzJp37LSU8asjE7jxTNp"
EXPECTED_BYTES = 28307958275
VIDEO_PATH = "/content/matches/JaHbat_vs_FC1_2026-09-13.mp4"
video_target = Path(VIDEO_PATH)
video_target.parent.mkdir(parents=True, exist_ok=True)
if not video_target.exists() or video_target.stat().st_size != EXPECTED_BYTES:
    assert shutil.disk_usage(video_target.parent).free > EXPECTED_BYTES + 2_000_000_000, "Not enough runtime disk space"
    partial = str(video_target) + ".partial"
    result = gdown.download(id=FILE_ID, output=partial, quiet=False, resume=True)
    assert result, "Drive requires authenticated access. Mount Drive and set VIDEO_PATH to the shared match before continuing."
    assert Path(partial).stat().st_size == EXPECTED_BYTES, "Download size mismatch"
    Path(partial).replace(video_target)
THRESHOLD = 0.5
BATCH_SIZE = 1
print("Match ready:", VIDEO_PATH)


Downloading...
From (original): https://drive.google.com/uc?id=1lkoCqWRF9CEvbzJp37LSU8asjE7jxTNp
From (redirected): https://drive.google.com/uc?id=1lkoCqWRF9CEvbzJp37LSU8asjE7jxTNp&confirm=t&uuid=989c86f8-8327-4f88-a76e-7c8a22e68754
To: /content/matches/JaHbat_vs_FC1_2026-09-13.mp4.partial
100%|██████████| 28.3G/28.3G [05:18<00:00, 88.8MB/s]

Match ready: /content/matches/JaHbat_vs_FC1_2026-09-13.mp4


In [ ]:
# Reacquire the TPU here because restarting the kernel clears notebook variables.
import os, sys, time, json
os.environ["TORCH_DEVICE_BACKEND_AUTOLOAD"] = "0"
from pathlib import Path
import torch
import torch_xla
import torch_xla.core.xla_model as xm
real_devices = torch_xla.real_devices()
assert len(real_devices) == 1 and str(real_devices[0]).startswith("TPU:"), (
    f"Connect a single-chip TPU v5e-1 kernel; found {real_devices}")
DEVICE = xm.xla_device()
assert DEVICE.type == "xla", f"Expected XLA device, got {DEVICE}"
assert (torch.ones(1, device=DEVICE) + 1).cpu().item() == 2, "TPU execution check failed"
accelerator = os.environ.get("TPU_ACCELERATOR_TYPE", "")
if accelerator:
    assert "v5e" in accelerator.lower() or "v5litepod-1" in accelerator.lower(), (
        f"Expected TPU v5e-1, got {accelerator}")
TPU = accelerator or str(real_devices[0])
assert VIDEO_PATH, "Set VIDEO_PATH to the full match first"
video = Path(VIDEO_PATH)
assert video.is_file(), f"Video not found: {video}"
sys.path.insert(0, str(REPO / "scripts"))
import evaluate_gar as eg  # Applies the Linux ffmpeg compatibility fix.
import numpy as np
from collections import Counter
tr = eg.tr
duration = tr.probe_duration(video)
assert duration > 0, "Video has no measurable duration"
print(f"Processing entire supplied match file: {duration / 60:.1f} minutes")
out = REPO / "outputs/full_match_tpu_v5e_1" / time.strftime("%Y%m%d_%H%M%S")
out.mkdir(parents=True)
last_print = [0.0]
def progress(fraction, message):
    now = time.monotonic()
    if now - last_print[0] >= 30 or fraction >= 1:
        print(message, flush=True)
        last_print[0] = now
started = time.perf_counter()
model_device = str(xm.xla_device())
model = tr.get_model(model_device)
scores = tr.compute_scores(video, model=model, start=0, duration=None,
                           batch_size=BATCH_SIZE, use_cache=False, progress=progress)
torch_xla.sync(wait=True)
elapsed = time.perf_counter() - started
assert np.isfinite(scores).all(), "Nonfinite model scores"
coverage = len(scores) / tr.SAMPLED_FPS
assert abs(coverage - duration) <= 2, f"Incomplete decode: {coverage} of {duration} seconds"
events = tr.extract_events(scores, threshold=THRESHOLD)
paths = tr.save_events(events, video, THRESHOLD, out / "events.json")
metrics = dict(tpu=TPU, device=model_device, revision=REVISION, video=video.name,
               duration_seconds=duration, processed_seconds=coverage,
               elapsed_seconds=elapsed, realtime_factor=duration / elapsed,
               batch_size=BATCH_SIZE, threshold=THRESHOLD, cached=False,
               event_count=len(events), per_class=dict(Counter(e['label'] for e in events)))
(out / "metrics.json").write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))
print("Results:", out)
